In [1]:
import pandas as pd

orders       = pd.read_csv("https://cdn.enqurious.com/documents/0518fd79-992c-420c-8580-a7acf31172b6_exorders.csv", parse_dates=["order_purchase_date"])
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1b6bff1-a6af-431b-bcc8-9618990bc5df_extransactions.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/604aa913-c904-4e6c-9d26-fdd443c52482_exproducts.csv")

# Step 1: Merge orders → transactions → products

# (check actual column name casing with df.columns before merging)

# Step 2: Extract month_num, month_name, season

# Step 3: Aggregate revenue by month × season × category

In [4]:
print(orders.columns,transactions.columns,products.columns)

Index(['order_id', 'customer_id', 'partner_id', 'ship_mode', 'order_status',
       'order_purchase_date', 'order_approved_at', 'order_dispatched_date',
       'order_delivered_date', 'order_estimated_delivery_date'],
      dtype='str') Index(['TransactionID', 'Order_ID', 'Product_ID', 'Sales_Amount', 'Quantity',
       'Discount', 'COGS'],
      dtype='str') Index(['product_id', 'product_name', 'colors', 'category', 'sub_category',
       'date_added', 'manufacturer', 'sizes', 'upc', 'weight',
       'product_photos_qty'],
      dtype='str')


In [14]:
transactions=transactions.drop_duplicates(subset=['Order_ID','Product_ID'])

In [16]:
merged=pd.merge(orders, transactions, how='left', left_on='order_id',right_on='Order_ID') 

In [17]:
merged=pd.merge(merged, products, how='left', left_on='Product_ID',right_on='product_id') 

In [18]:
merged.head(5)

,order_id,customer_id,partner_id,ship_mode,order_status,order_purchase_date,order_approved_at,order_dispatched_date,order_delivered_date,order_estimated_delivery_date,...,product_name,colors,category,sub_category,date_added,manufacturer,sizes,upc,weight,product_photos_qty
0,CA-2014-100006,MH-17785,VEN02,Standard,delivered,2018-05-11 20:07:00,2018-05-11 20:30:00.000,2018-05-16 15:14:00.000,2018-05-19 13:42:00.000,2018-05-23,...,GBC Recycled Grain Textured Covers,Blue,Office Supplies,Binders,2017-01-23,Callisto,"7,6",8.02E+11,NaN,0
1,CA-2014-100090,PC-18745,VEN01,Standard,delivered,2018-01-30 10:21:00,2018-01-30 10:35:00.000,2018-01-31 20:39:00.000,2018-02-14 23:39:00.000,2018-02-26,...,Target Practical Foundations 30 x 60 Training ...,Pink,Furniture,Tables,2016-10-05,Polo Ralph Lauren,NaN,8.90E+11,NaN,2
2,CA-2014-100090,PC-18745,VEN01,Standard,delivered,2018-01-30 10:21:00,2018-01-30 10:35:00.000,2018-01-31 20:39:00.000,2018-02-14 23:39:00.000,2018-02-26,...,Polycom CX600 IP Phone VoIP phone,Blue,Technology,Phones,2017-01-12,Hi-Tec,"8 M,15 M,10 M,7.5 M,7 M,10.5 M,8 W,10.5 W,8.5 ...",90641331434,NaN,0
3,CA-2014-100293,MV-18190,VEN01,Standard,delivered,2017-10-04 14:15:00,2017-10-04 15:25:00.000,2017-10-06 18:27:00.000,2017-10-11 14:50:00.000,2017-10-27,...,Newell 319,Pink,Office Supplies,Art,2015-12-01,Invicta,NaN,7.23E+11,2.0 lbs,0
4,CA-2014-100328,AH-10195,VEN02,Standard,delivered,2018-05-17 14:22:00,2018-05-18 00:37:00.000,2018-05-18 13:59:00.000,2018-06-06 12:55:00.000,2018-06-08,...,Memorex Micro Travel Drive 16 GB,Blue,Technology,Accessories,2016-11-11,DbDk Fashion,"5.5,7,6,6.5,7.5,8",NaN,NaN,4


In [22]:
merged['month_num']=merged['order_purchase_date'].dt.month
merged['month_name']=merged['order_purchase_date'].dt.month_name()
merged['season'] = merged['month_num'].map({12: 'Winter', 1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring', 6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Autumn', 10: 'Autumn', 11: 'Autumn'})

In [27]:
rep=merged.groupby(['month_num','month_name','season','category'])['Sales_Amount'].sum().reset_index().sort_values(by=['month_num','category'])

In [28]:
rep 

,month_num,month_name,season,category,Sales_Amount
0,1,January,Winter,Furniture,62695.5571
1,1,January,Winter,Office Supplies,58986.6470
2,1,January,Winter,Technology,53778.6280
3,2,February,Winter,Furniture,58640.1918
4,2,February,Winter,Office Supplies,64841.3870
5,2,February,Winter,Technology,73089.5460
6,3,March,Spring,Furniture,73210.6145
7,3,March,Spring,Office Supplies,86406.2450
8,3,March,Spring,Technology,100605.1070
9,4,April,Spring,Furniture,72294.2453


In [33]:
piv_rep=pd.pivot_table(rep,index=['month_num', 'month_name', 'season'], columns='category', values='Sales_Amount', aggfunc='sum',fill_value=0).reset_index() 
piv_rep.columns.name = None


In [35]:
unpiv_rep = pd.melt(piv_rep, id_vars=['month_num', 'month_name', 'season'], var_name='category', value_name='revenue')

In [37]:
season_ranked = (
    unpiv_rep
    .groupby(['season', 'category'], as_index=False)['revenue']
    .sum()
    .assign(
        category_rank=lambda df: df.groupby('season')['revenue']
        .rank(method='dense', ascending=False)
        .astype(int)
    )
    .sort_values(['season', 'category_rank'])
)

season_ranked

,season,category,revenue,category_rank
1,Autumn,Office Supplies,123870.0910,1
0,Autumn,Furniture,122782.8050,2
2,Autumn,Technology,122298.1460,3
5,Spring,Technology,286617.0840,1
4,Spring,Office Supplies,232440.9780,2
3,Spring,Furniture,220116.8008,3
8,Summer,Technology,229387.5150,1
6,Summer,Furniture,220157.0793,2
7,Summer,Office Supplies,201348.3840,3
9,Winter,Furniture,167225.0617,1
